In [3]:
!pip install pandas scikit-learn datasets

import pandas as pd
import numpy as np
import re

from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split

In [4]:
import os

token = os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACE_TOKEN") or os.getenv("HF_HUB_TOKEN")

dataset = load_dataset("jquigl/imdb-genres", token=token) if token else load_dataset("jquigl/imdb-genres")

df = pd.DataFrame(dataset["train"])

# Normalize source column names to the names used downstream.
df = df.rename(
    columns={
        "movie title - year": "title",
        "expanded-genres": "genres",
    }
)

# Keep only needed columns
required_cols = ["title", "description", "genres", "rating"]
df = df[required_cols]

# Drop missing values
df = df.dropna(subset=["description", "title"])
df.reset_index(drop=True, inplace=True)

df.head()

,title,description,genres,rating
0,Flaming Ears - 1992,Flaming Ears is a pop sci-fi lesbian fantasy f...,"Fantasy, Sci-Fi",6.0
1,Jeg elsker dig - 1957,Six people - three couples - meet at random at...,"Comedy, Drama, Romance",5.8
2,Povjerenje - 2021,"In a small unnamed town, in year 2025, Krsto w...",Thriller,NaN
3,Gulliver Returns - 2021,The legendary Gulliver returns to the Kingdom ...,"Animation, Adventure, Family",4.4
4,Prithvi Vallabh - 1924,"Seminal silent historical film, the story feat...","Biography, Drama, Romance",NaN


In [5]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

stop_words = set(ENGLISH_STOP_WORDS)

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # remove punctuation
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)

df['clean_description'] = df['description'].apply(clean_text)

df[['description', 'clean_description']].head()

,description,clean_description
0,Flaming Ears is a pop sci-fi lesbian fantasy f...,flaming ears pop scifi lesbian fantasy feature...
1,Six people - three couples - meet at random at...,people couples meet random dance restaurant co...
2,"In a small unnamed town, in year 2025, Krsto w...",small unnamed town year krsto works agency off...
3,The legendary Gulliver returns to the Kingdom ...,legendary gulliver returns kingdom lilliput gi...
4,"Seminal silent historical film, the story feat...",seminal silent historical film story features ...


In [6]:
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

# Reset indexes so cosine matrix positions align with DataFrame row positions.
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

In [7]:
vectorizer = TfidfVectorizer(max_features=5000)
X_train = vectorizer.fit_transform(train_df['clean_description'])

In [8]:
# Avoid materializing a huge NxN matrix in memory.
# Similarity is computed on-demand in recommend_movies().

In [9]:
indices = (
    train_df.reset_index()
    .drop_duplicates(subset=["title"])
    .set_index("title")["index"]
    .astype(int)
)

def recommend_movies(title, top_n=5):
    if title not in indices:
        return "Movie not found."

    idx = int(indices[title])

    # Compute similarity only for the selected movie row.
    sim_scores = cosine_similarity(X_train[idx], X_train)[0]
    sim_scores = list(enumerate(sim_scores))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    sim_scores = sim_scores[1 : top_n + 1]

    movie_indices = [i[0] for i in sim_scores]

    return train_df.iloc[movie_indices][["title", "genres", "rating"]]

In [10]:
def recommend_from_description(desc, top_n=5):
    desc_clean = clean_text(desc)
    desc_vec = vectorizer.transform([desc_clean])

    sim_scores = cosine_similarity(desc_vec, X_train)
    sim_scores = list(enumerate(sim_scores[0]))

    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[:top_n]

    movie_indices = [i[0] for i in sim_scores]

    return train_df.iloc[movie_indices][["title", "genres", "rating"]]

In [11]:
# By movie title from this dataset
print(recommend_movies("Avatar - 2009"))

# By vague description
print(recommend_from_description("space adventure with aliens and futuristic battles"))

                        title                      genres  rating
40633           Avatar - 2009  Action, Adventure, Fantasy     7.9
23212   Galit sa mundo - 1989               Action, Drama     NaN
132416         Unlucky - 2011     Comedy, Fantasy, Sci-Fi     5.6
167473          TuHiRe - 2015              Drama, Romance     6.3
147165          Dark Moon - I            Horror, Thriller     NaN
                           title                     genres  rating
173026              Impulse - II                     Action     NaN
38057   5000 Space Aliens - 2021  Animation, Comedy, Sci-Fi     NaN
170456  5000 Space Aliens - 2021  Animation, Comedy, Sci-Fi     NaN
134031      The Last Ride - 1931               Crime, Drama     NaN
88993    Panda vs. Aliens - 2021  Animation, Comedy, Family     3.5
